> **Superseded.** The folds in this notebook were made by segment. They were later replaced by folds made by twin group (see notebook 06). The saved files in `splits/` now hold the corrected folds.

In [1]:
import sys
sys.path.append("../..")

import pandas as pd

from src.config import PROCESSED, SPLITS, N_REPEATS
from src.cv import make_fold_table, fold_of_rows

dev = pd.read_csv(PROCESSED / "dev_2v3.csv")
hold_ids = set(pd.read_csv(PROCESSED / "holdout_2v3.csv", usecols=["segment"])["segment"])   # IDs only

table = make_fold_table(dev)
print(table.shape)          
table.head()

(160, 7)


,segment,y,repeat_0,repeat_1,repeat_2,repeat_3,repeat_4
0,V1.102,3,3,2,6,6,5
1,V1.103,2,6,5,0,4,5
2,V1.112,3,0,4,5,3,9
3,V1.113,3,2,8,4,1,4
4,V1.114,2,3,3,4,0,3


In [2]:
assert table["segment"].is_unique and len(table) == 160  
assert set(table["segment"]).isdisjoint(hold_ids) 
for r in range(N_REPEATS):
    counts = table.groupby([f"repeat_{r}", "y"]).size().unstack()
    assert (counts == 8).all().all() 
print("all checks passed")
print("same fold in repeat 0 and repeat 1:", round((table["repeat_0"] == table["repeat_1"]).mean(), 2))

rows_fold = fold_of_rows(dev, table, 0)
print("chunks per fold:", pd.Series(rows_fold).value_counts().unique()) 

all checks passed
same fold in repeat 0 and repeat 1: 0.08
chunks per fold: [368]


In [3]:
SPLITS.mkdir(exist_ok=True)
if (SPLITS / "cv_folds.csv").exists():
    print("cv_folds.csv already exists, not overwriting")
else:
    table.to_csv(SPLITS / "cv_folds.csv", index=False)
    pd.DataFrame({"segment": sorted(hold_ids)}).to_csv(SPLITS / "holdout_segments.csv", index=False)
    print("saved")

saved


# 03 Evaluation design and CV folds

- **Unit of prediction: the segment.** Each segment's 23 chunk probabilities are averaged into one prediction. (The question is about a recording, and chunks of one recording are not independent.)
- **Folds: 10 outer folds on segments, stratified.** 16 segments per fold (8 per class). All 23 chunks of a segment stay in the same fold.
- **Repeats: 5 different shuffles when comparing models.** One nested 10-fold (5-fold inner loop) for the final tuned model.
- **Scores: ROC-AUC is the headline.** It is pooled over out-of-fold segments and averaged over repeats. Accuracy at 0.5 and the confusion matrix (class 2 = positive) are shown beside it. ROC-AUC is also used to choose model settings in the inner loop.
- **Hold-out: 40 segments, used once at the end.**
- **Folds are fixed and saved** in `splits/cv_folds.csv`, and the hold-out segment IDs are in `splits/holdout_segments.csv`. Every model uses the same folds.

## Checks passed
- 160 development segments, each listed once, none from the hold-out.
- Every fold has 8 class-2 and 8 class-3 segments, in all 5 repeats.
- The repeats differ: only 8% of segments are in the same fold in repeat 0 and repeat 1.
- Each fold holds 368 chunks (16 x 23), and no segment is split across folds.

## Limits
- With 160 segments, scores are noisy. Report ranges, not single numbers.
- Grouping by segment prevents chunk leakage. The data has no patient ID, so patient overlap cannot be ruled out.